In [68]:
import pandas as pd
import numpy as np

df = pd.read_csv("rahul_transactions.csv")

print(df.head())
print(df.shape)
print(df.columns)

          Date   Time                       Description   Type  Amount  \
0   2024-01-01  03:11                AMAZON SELLER SVCS  Debit   ₹2462   
1    01-Jan-24  05:44                         BHIM-BMTC     DR   50.00   
2    01-Jan-24  09:35  NEFT-TECHCRUSH LABS-SALARY MAY24     CR  ₹84728   
3   2024-01-01  14:07              UPI-AMAN-8934@OKAXIS  Debit   ₹1828   
4  01 Jan 2024  14:23                      BHIM-BLINKIT  Debit  270.00   

    Balance  Mode        Ref  
0  678275.0   UPI  TXN190872  
1  681007.0   UPI  TXN143064  
2  484728.0  NEFT  TXN246316  
3 -748745.0   UPI  TXN569226  
4  680737.0   UPI  TXN968962  
(1328, 8)
Index(['Date', 'Time', 'Description', 'Type', 'Amount', 'Balance', 'Mode',
       'Ref'],
      dtype='object')


In [69]:
df["date"] = pd.to_datetime(
    df["Date"],
    errors="coerce",
    dayfirst=True
)

df["money"] = df["Amount"].astype(str)
df["money"] = df["money"].str.replace("₹", "", regex=False)
df["money"] = df["money"].str.replace("Rs.", "", regex=False)
df["money"] = df["money"].str.replace(",", "", regex=False)
df["money"] = df["money"].str.strip()

df["money"] = pd.to_numeric(
    df["money"],
    errors="coerce"
)

df["kind"] = df["Type"].astype(str)
df["kind"] = df["kind"].str.lower()
df["kind"] = df["kind"].str.replace("dr", "debit", regex=False)
df["kind"] = df["kind"].str.replace("cr", "credit", regex=False)

df["Mode"] = df["Mode"].replace("", np.nan)

old = len(df)

df = df.drop_duplicates()

new = len(df)

removed = old - new

bad_date = df["date"].isna().sum()
bad_money = df["money"].isna().sum()

df = df.dropna(subset=["date", "money"])

print("Transactions after cleaning:", len(df))
print("Duplicates removed:", removed)
print("Bad dates:", bad_date)
print("Bad amounts:", bad_money)

Transactions after cleaning: 143
Duplicates removed: 18
Bad dates: 1167
Bad amounts: 0


In [70]:
print(df["Description"].unique())

['AMAZON SELLER SVCS' 'UPI-AMAN-8934@OKAXIS' 'BHIM ZEPTO'
 'UPI-UBER-2426@HDFCBANK' 'POS SWIGGY BANGALORE' 'ANI Technologies'
 'POS SWIGGY-RESTAURANT' 'POS UBER BANGALORE' 'TWC INDIA'
 'UPI-STARBUCKS@AXIS' 'POS BLINKIT' 'ZOMATO MEDIA P L' 'BHIM SWIGGY'
 'OLA CABS' 'UPI-DMART@OKAXIS' 'POS ZEPTO BANGALORE' 'UPI-MYNTRA@HDFCBANK'
 'BANGALORE ELEC SUPPLY' 'Amazon Pay India' 'UPI-PRIYA-2221@OKAXIS'
 'INSTAMART BANGALORE' 'UPI-ZOMATO-LIMITED@PAYTM'
 'UPI-SWIGGY-7959@HDFCBANK' 'UPI-SWIGGY-INSTAMART@OKAXIS' 'POS OLA-PRIME'
 'POS EMPIRE RESTAURANT' 'SWIGGY-INSTAMART' 'POS NYKAA BANGALORE'
 'ROPPEN TRANSPORTATION' 'UPI-NYKAA@AXIS' 'UPI-SWIGGY4619@OKAXIS'
 'UPI-FLIPKART@HDFCBANK' 'IMPS ZERODHA-COIN' 'UPI-THIRDWAVE@OKAXIS'
 'UPI-SWIGGY-1309@HDFCBANK' 'NEFT-TECHCRUSH LABS-SALARY MAY24'
 'BHIM-BLINKIT' 'UPI-ZEPTO-4547@HDFCBANK' 'RAPIDO BIKE TAXI'
 'UPI-RESTAURANT-6377@PAYTM' 'Swiggy*Order' 'COFFEE DAY GLOBAL'
 'FLIPKART INDIA' 'UPI-CCD@HDFCBANK' 'UPI-VIKAS-5416@OKAXIS'
 'UPI-BOOKMYSHOW@HDFCBANK' 'UPI

In [71]:
# AI-assisted: Used AI to help understand and debug the code.
vendors = {
    "Amazon Pay": ["AMAZON PAY"],
    "Amazon": ["AMAZON", "AMZN"],
    "Zepto": ["ZEPTO"],
    "Uber": ["UBER"],
    "Swiggy": ["SWIGGY", "BUNDL"],
    "ANI Technologies": ["ANI TECHNOLOGIES"],
    "Third Wave Coffee": ["TWC INDIA", "THIRDWAVE", "THIRD WAVE COFFEE"],
    "Starbucks": ["STARBUCKS"],
    "Blinkit": ["BLINKIT"],
    "Zomato": ["ZOMATO"],
    "Ola Electric": ["OLA ELECTRIC"],
    "DMart": ["DMART"],
    "Myntra": ["MYNTRA"],
    "Bangalore Electricity Supply": ["BANGALORE ELEC SUPPLY"],
    "Instamart": ["INSTAMART"],
    "Empire Restaurant": ["EMPIRE RESTAURANT"],
    "Nykaa": ["NYKAA"],
    "Roppen": ["ROPPEN"],
    "Flipkart": ["FLIPKART"],
    "Zerodha": ["ZERODHA"],
    "Salary": ["TECHCRUSH LABS-SALARY"],
    "Rapido": ["RAPIDO"],
    "Cafe Coffee Day": ["COFFEE DAY", "CCD"],
    "BookMyShow": ["BOOKMYSHOW", "BMS MOVIE"],
    "Hotstar": ["HOTSTAR"],
    "Netflix": ["NETFLIX"],
    "Restaurant": ["RESTAURANT"],
    "Innovative Retail": ["INNOVATIVE RETAIL"],
    "Avenue Supermarts": ["AVENUE SUPERMARTS"],
    "BigBasket": ["BIGBASKET"],
    "HP Petrol": ["HP PETROL"],
    "Ola": ["OLA"],
    "KiranaKart": ["KIRANAKART"],
    "Rent": ["RENT-LANDLORD"],
    "Groww": ["GROWWPAY"],
    "VI": ["VI-RECHARGE"],
    "P2P Transfer": [],
    "Cash Withdrawal": []
}

In [72]:
def get_vendor(text):
    text = str(text).upper()

    if "ATM-WDL" in text:
        return "Cash Withdrawal"

    if text.startswith("UPI-") and "@" in text:
        parts = text.split("-")

        if len(parts) >= 2:
            person = parts[1]

            found = False

            for name in vendors:
                for word in vendors[name]:
                    if word.upper() in text:
                        found = True
                        break

                if found:
                    break

            if found == False:
                return "P2P Transfer"

    for name in vendors:
        words = vendors[name]

        for word in words:
            if word.upper() in text:
                return name

    return "Uncategorised"

In [73]:
df["vendor"] = df["Description"].apply(get_vendor)

print("Number of vendors:", df["vendor"].nunique())
print()

print(df["vendor"].value_counts().head(10))

Number of vendors: 38

vendor
Swiggy               28
Zomato               16
Uber                  7
Ola                   6
Zepto                 5
ANI Technologies      5
Third Wave Coffee     5
P2P Transfer          4
BookMyShow            4
Starbucks             4
Name: count, dtype: int64


In [74]:
bad = df[df["vendor"] == "Uncategorised"]

print("Uncategorised transactions:", len(bad))
print()

print(bad["Description"].unique())

Uncategorised transactions: 0

[]


In [75]:
# AI-assisted: Used AI to help understand and debug the code.
cats = {
    "Swiggy": "Food Delivery",
    "Zomato": "Food Delivery",

    "Zepto": "Quick Commerce",
    "Blinkit": "Quick Commerce",
    "Instamart": "Quick Commerce",

    "Amazon": "E-commerce",
    "Amazon Pay": "E-commerce",
    "Flipkart": "E-commerce",
    "Myntra": "E-commerce",
    "Nykaa": "E-commerce",

    "Uber": "Transport",
    "Ola": "Transport",
    "Rapido": "Transport",
    "Roppen": "Transport",

    "Starbucks": "Cafe",
    "Third Wave Coffee": "Cafe",
    "Cafe Coffee Day": "Cafe",

    "Restaurant": "Restaurants",

    "Netflix": "Subscriptions",
    "Hotstar": "Subscriptions",
    "VI": "Subscriptions",

    "BESCOM": "Utilities",
    "Rent": "Utilities",

    "DMart": "Groceries",
    "BigBasket": "Groceries",
    "KiranaKart": "Groceries",

    "Zerodha": "Investments",
    "Groww": "Investments",

    "HP Petrol": "Fuel",

    "BookMyShow": "Entertainment",

    "P2P Transfer": "Personal Transfer",
    "Cash Withdrawal": "Cash Withdrawal",
    "ANI Technologies": "Transport",
    "Bangalore Electricity Supply": "Utilities",
    "Empire Restaurant": "Restaurants",
    "Salary": "Personal Transfer",
    "Innovative Retail": "E-commerce",
    "Third Wave Coffee": "Cafe",
    "Avenue Supermarts": "Groceries"
}

In [76]:
df["category"] = df["vendor"].map(cats)

df.loc[df["vendor"] == "P2P Transfer", "category"] = "Personal Transfer"
df.loc[df["vendor"] == "Cash Withdrawal", "category"] = "Cash Withdrawal"

df["category"] = df["category"].fillna("Uncategorised")

print(df["category"].value_counts())

category
Food Delivery        44
Transport            21
E-commerce           14
Cafe                 12
Quick Commerce       10
Groceries             7
Restaurants           7
Personal Transfer     6
Investments           5
Utilities             4
Entertainment         4
Subscriptions         3
Cash Withdrawal       3
Uncategorised         2
Fuel                  1
Name: count, dtype: int64


In [77]:
credit = df[df["kind"] == "credit"]
debit = df[df["kind"] == "debit"]

total_in = credit["money"].sum()
total_out = debit["money"].sum()

change = total_in - total_out

if total_in != 0:
    save_rate = (change / total_in) * 100
else:
    save_rate = 0

print("=" * 60)
print("                 SPENDDNA REPORT")
print("=" * 60)

print("Total credits :", f"Rs. {total_in:,.2f}")
print("Total debits  :", f"Rs. {total_out:,.2f}")
print("Net change    :", f"Rs. {change:,.2f}")
print("Savings rate  :", f"{save_rate:.2f}%")
print("Transactions  :", len(df))
print("Vendors       :", df["vendor"].nunique())

                 SPENDDNA REPORT
Total credits : Rs. 170,094.00
Total debits  : Rs. 205,310.00
Net change    : Rs. -35,216.00
Savings rate  : -20.70%
Transactions  : 143
Vendors       : 38


In [78]:
cat_total = debit.groupby(
    "category"
)["money"].sum()

cat_total = cat_total.sort_values(
    ascending=False
)

print()
print("=" * 60)
print("                 TOP CATEGORIES")
print("=" * 60)

for name in cat_total.head(5).index:

    value = cat_total[name]

    if total_out != 0:
        percent = (value / total_out) * 100
    else:
        percent = 0

    bars = int(percent)

    print(
        f"{name:<20} "
        f"{'#' * bars:<25} "
        f"{percent:6.2f}%  "
        f"Rs. {value:,.2f}"
    )


                 TOP CATEGORIES
Investments          ###############################  31.41%  Rs. 64,496.00
Utilities            ###################        19.32%  Rs. 39,673.00
E-commerce           ##################         18.11%  Rs. 37,182.00
Food Delivery        #########                   9.55%  Rs. 19,617.00
Groceries            ###                         3.99%  Rs. 8,193.00


In [79]:
ven_total = debit.groupby(
    "vendor"
)["money"].sum()

ven_total = ven_total.sort_values(
    ascending=False
)

print()
print("=" * 60)
print("                   TOP VENDORS")
print("=" * 60)

for name in ven_total.head(5).index:

    value = ven_total[name]

    count = len(
        debit[debit["vendor"] == name]
    )

    print(
        f"{name:<20} "
        f"Rs. {value:>12,.2f}   "
        f"{count:>4} transactions"
    )


                   TOP VENDORS
Zerodha              Rs.    60,000.00      4 transactions
Rent                 Rs.    36,000.00      2 transactions
Myntra               Rs.    13,629.00      2 transactions
Swiggy               Rs.    12,638.00     28 transactions
Flipkart             Rs.    10,041.00      4 transactions


In [80]:
df["month"] = df["date"].dt.month

month_data = df[
    df["kind"] == "debit"
]

month_table = month_data.pivot_table(
    values="money",
    index="category",
    columns="month",
    aggfunc="sum"
)

month_table = month_table.fillna(0)

print("=" * 60)
print("                 MONTHLY SPENDING")
print("=" * 60)

print(month_table)
print()
print("=" * 60)
print("             JANUARY TO JUNE CHANGE")
print("=" * 60)

changes = {}

for name in month_table.index:

    first = month_table.loc[name, 1]
    last = month_table.loc[name, 6]

    if first != 0:
        percent = ((last - first) / first) * 100
        changes[name] = percent

        print(
            f"{name:<20} {percent:>8.2f}%"
        )

                 MONTHLY SPENDING
month                  1       2        3       4        5       6        7   \
category                                                                       
Cafe                295.0   355.0      0.0   960.0    166.0   442.0      0.0   
Cash Withdrawal       0.0     0.0      0.0     0.0      0.0     0.0      0.0   
E-commerce         6972.0   996.0   2147.0  3311.0      0.0  4868.0   2425.0   
Entertainment         0.0     0.0      0.0   343.0      0.0     0.0    562.0   
Food Delivery      1180.0  3454.0   2862.0   632.0   1159.0  1636.0   1686.0   
Fuel                  0.0     0.0      0.0     0.0      0.0     0.0      0.0   
Groceries             0.0     0.0   1382.0     0.0      0.0   352.0   1257.0   
Investments           0.0     0.0      0.0     0.0      0.0     0.0  15000.0   
Personal Transfer  1828.0     0.0      0.0     0.0      0.0  1860.0      0.0   
Quick Commerce     1616.0   697.0      0.0     0.0    502.0     0.0    510.0   
Restau

In [81]:
# AI-assisted: Used AI to help understand and debug the code.

spending = debit[
    debit["category"] != "Personal Transfer"
].copy()

cat_money = spending.groupby(
    "category"
)["money"].sum()

total_spending = spending["money"].sum()

cat_percent = (
    cat_money / total_spending
) * 100

print()
print("=" * 60)
print("              CATEGORY PERCENTAGES")
print("=" * 60)

for name in cat_percent.sort_values(
    ascending=False
).index:

    print(
        f"{name:<20} "
        f"{cat_percent[name]:>7.2f}%"
    )


              CATEGORY PERCENTAGES
Investments            32.23%
Utilities              19.83%
E-commerce             18.58%
Food Delivery           9.80%
Groceries               4.09%
Restaurants             3.34%
Transport               3.16%
Quick Commerce          2.39%
Cafe                    1.99%
Cash Withdrawal         1.50%
Subscriptions           1.20%
Entertainment           1.18%
Fuel                    0.54%
Uncategorised           0.16%


In [82]:
df["hour"] = df["Time"].str[:2].astype(int)

debit = df[df["kind"] == "debit"].copy()

time_total = debit.groupby(
    ["category", "hour"]
)["money"].sum()

print("=" * 60)
print("                TIME OF DAY SPENDING")
print("=" * 60)

for name in debit["category"].unique():

    print()
    print(name)

    for h in range(24):

        key = (name, h)

        if key in time_total.index:

            value = time_total.loc[key]

            bars = int(value / 1000)

            print(
                f"{h:02d}:00 | {'#' * bars}"
            )

                TIME OF DAY SPENDING

E-commerce
01:00 | 
03:00 | ###
06:00 | #
07:00 | 
11:00 | ###
13:00 | 
15:00 | #
18:00 | ######
19:00 | ####
20:00 | ##########
21:00 | ###

Personal Transfer
14:00 | #
16:00 | 
21:00 | #
22:00 | #

Quick Commerce
02:00 | 
08:00 | 
09:00 | 
10:00 | 
14:00 | #
17:00 | 
18:00 | 
23:00 | 

Transport
00:00 | 
05:00 | 
06:00 | 
07:00 | 
09:00 | 
10:00 | 
13:00 | 
14:00 | 
15:00 | 
16:00 | 
18:00 | 
20:00 | 
21:00 | 

Food Delivery
00:00 | 
02:00 | 
04:00 | 
05:00 | 
08:00 | 
09:00 | 
11:00 | ##
12:00 | ##
13:00 | 
14:00 | 
15:00 | #
16:00 | 
17:00 | 
18:00 | #
19:00 | 
20:00 | #
21:00 | ##
22:00 | #
23:00 | 

Cafe
00:00 | 
08:00 | 
09:00 | 
10:00 | #
14:00 | 
16:00 | 
17:00 | 

Groceries
08:00 | ##
11:00 | ###
16:00 | 
18:00 | #
20:00 | #

Utilities
13:00 | ##################
14:00 | ##################
17:00 | #
18:00 | ##

Restaurants
10:00 | 
11:00 | 
13:00 | ##
18:00 | #
20:00 | ##

Investments
04:00 | ###############
05:00 | ####
10:00 | ##########

In [83]:
food = debit[
    debit["category"] == "Food Delivery"
].copy()

late_food = food[
    (food["hour"] >= 21) |
    (food["hour"] <= 2)
]

if len(food) != 0:
    late_percent = (
        len(late_food) / len(food)
    ) * 100
else:
    late_percent = 0

print()
print("=" * 60)
print("              LATE NIGHT FOOD")
print("=" * 60)

print(
    "Food Delivery transactions:",
    len(food)
)

print(
    "Late night transactions:",
    len(late_food)
)

print(
    f"Late night percentage: {late_percent:.2f}%"
)


              LATE NIGHT FOOD
Food Delivery transactions: 44
Late night transactions: 11
Late night percentage: 25.00%


In [84]:
mean_value = debit.groupby(
    "category"
)["money"].transform("mean")

std_value = debit.groupby(
    "category"
)["money"].transform("std")

debit["z"] = (
    debit["money"] - mean_value
) / std_value

odd = debit[
    debit["z"] > 2
].copy()

odd = odd.sort_values(
    "z",
    ascending=False
)

print()
print("=" * 60)
print("                  ANOMALIES")
print("=" * 60)

print(
    "Number of anomalies:",
    len(odd)
)

print()

for i, row in odd.head(5).iterrows():

    print(
        f"{row['date'].date()} | "
        f"{row['vendor']:<18} | "
        f"{row['category']:<18} | "
        f"Rs. {row['money']:>10,.2f} | "
        f"z = {row['z']:.2f}"
    )


                  ANOMALIES
Number of anomalies: 1

2024-09-01 | Myntra             | E-commerce         | Rs.  10,745.00 | z = 3.18


In [85]:
def foodie():
    a = cat_percent.get(
        "Food Delivery", 0
    )

    b = cat_percent.get(
        "Restaurants", 0
    )

    c = cat_percent.get(
        "Cafe", 0
    )

    return a + b + c > 25


def quick_person():
    a = cat_percent.get(
        "Quick Commerce", 0
    )

    return a > 15


def shop_person():
    a = cat_percent.get(
        "E-commerce", 0
    )

    return a > 15


def investor():
    a = cat_percent.get(
        "Investments", 0
    )

    return a > 15


def night_food():
    food_rows = debit[
        debit["category"] == "Food Delivery"
    ]

    if len(food_rows) == 0:
        return False

    night_rows = food_rows[
        (food_rows["hour"] >= 21) |
        (food_rows["hour"] <= 2)
    ]

    percent = (
        len(night_rows) /
        len(food_rows)
    ) * 100

    return percent > 50


def cab_person():
    a = cat_percent.get(
        "Transport", 0
    )

    return a > 10


def sub_person():
    sub = df[
        df["category"] == "Subscriptions"
    ]

    number = sub["vendor"].nunique()

    return number >= 5


def yolo_person():
    return save_rate < 10


def saver_person():
    return save_rate > 40

In [86]:
people = []

if foodie():
    people.append("THE FOODIE")

if quick_person():
    people.append("THE QUICK COMMERCE JUNKIE")

if shop_person():
    people.append("THE SHOPAHOLIC")

if investor():
    people.append("THE INVESTOR")

if night_food():
    people.append("THE LATE-NIGHT SNACKER")

if cab_person():
    people.append("THE CAB COMMUTER")

if sub_person():
    people.append("THE SUBSCRIPTION LOVER")

if yolo_person():
    people.append("THE YOLO SPENDER")

if saver_person():
    people.append("THE DISCIPLINED SAVER")

print()
print("=" * 60)
print("                SPENDING ARCHETYPES")
print("=" * 60)

for person in people:
    print("->", person)

print()
print("Savings rate:", f"{save_rate:.2f}%")


                SPENDING ARCHETYPES
-> THE SHOPAHOLIC
-> THE INVESTOR
-> THE YOLO SPENDER

Savings rate: -20.70%


In [87]:
print()
print("=" * 65)
print("                    SPENDDNA REPORT")
print("                    RAHUL SHARMA")
print("=" * 65)

print()
print("EXECUTIVE SUMMARY")
print("-" * 65)

print(
    f"Total credits       : Rs. {total_in:,.2f}"
)

print(
    f"Total debits        : Rs. {total_out:,.2f}"
)

print(
    f"Net change          : Rs. {change:,.2f}"
)

print(
    f"Savings rate        : {save_rate:.2f}%"
)

print(
    f"Transactions        : {len(df)}"
)

print(
    f"Unique vendors      : {df['vendor'].nunique()}"
)

print()
print("TOP CATEGORIES")
print("-" * 65)

for name in cat_total.head(5).index:

    value = cat_total[name]
    percent = (value / total_out) * 100

    print(
        f"{name:<20} "
        f"{percent:>7.2f}%   "
        f"Rs. {value:,.2f}"
    )

print()
print("TOP VENDORS")
print("-" * 65)

for name in ven_total.head(5).index:

    value = ven_total[name]

    print(
        f"{name:<20} "
        f"Rs. {value:,.2f}"
    )
# AI-assisted: Used AI to help understand and debug the code.
print()
print("ARCHETYPES")
print("-" * 65)

for person in people:
    print("->", person)

print()
print("ANOMALIES")
print("-" * 65)

for i, row in odd.head(5).iterrows():

    print(
        f"{row['date'].date()} | "
        f"{row['vendor']:<18} | "
        f"Rs. {row['money']:,.2f} | "
        f"z = {row['z']:.2f}"
    )

print()
print("=" * 65)
print("                    END OF REPORT")
print("=" * 65)


                    SPENDDNA REPORT
                    RAHUL SHARMA

EXECUTIVE SUMMARY
-----------------------------------------------------------------
Total credits       : Rs. 170,094.00
Total debits        : Rs. 205,310.00
Net change          : Rs. -35,216.00
Savings rate        : -20.70%
Transactions        : 143
Unique vendors      : 38

TOP CATEGORIES
-----------------------------------------------------------------
Investments            31.41%   Rs. 64,496.00
Utilities              19.32%   Rs. 39,673.00
E-commerce             18.11%   Rs. 37,182.00
Food Delivery           9.55%   Rs. 19,617.00
Groceries               3.99%   Rs. 8,193.00

TOP VENDORS
-----------------------------------------------------------------
Zerodha              Rs. 60,000.00
Rent                 Rs. 36,000.00
Myntra               Rs. 13,629.00
Swiggy               Rs. 12,638.00
Flipkart             Rs. 10,041.00

ARCHETYPES
-----------------------------------------------------------------
-> THE SHO

AI assistance disclosure: AI was used for syntax guidance, code structuring, and debugging. The vendor mapping, category mapping, and final data-specific observations were created and verified by me using the provided dataset.